## I. Small dataset creation

In [ ]:
a = np.random.randint(-10, 10, (4, 3))
b = np.random.randint(-10, 10, (4, 3))

np.stack([a, b], axis=2).shape

In [ ]:
import numpy as np # numeric python, численные 
import pandas as pd # data analysis

living_area     = np.random.randint(30, 200, 5) # random integer от 30 до 200 в кол-ве 5
time_to_subway  = np.random.randint(4, 30, 5)
age_of_building = np.random.randint(0, 60, 5)

In [ ]:
dataset = pd.DataFrame(np.stack(
    [living_area, time_to_subway, age_of_building], 
    axis=1
), columns=['living_area', 'time_to_subway', 'age_of_building'])

dataset

In [ ]:
y_true = np.array([40, 53, 65, 30, 33])

## II. Dataset creation

In [ ]:
print(np.array([10, 20, 30]) * np.array([7, 8, 9]))
print(np.array([10, 20, 30]) @ np.array([7, 8, 9]) + 3)
print(np.array([10, 20, 30, 1]) @ np.array([7, 8, 9, 3]))

In [ ]:
# x_features = [10, 20, 30]
# weights = [7, 8, 9]
# bias = +3
# x_features @ weights + bias

# решение
# x_features = [10, 20, 30, 1] - добавили новый единичный признак всем
# weights = [7, 8, 9, 3] - вписали bias в вектор весов

number_of_objects  = 2000 # кол-во объектов
number_of_features = 5 # кол-во признаков

X = np.random.randint(-20, 20, (number_of_objects, number_of_features)) # массив объекты - признаки
X = np.concatenate([X, np.ones((number_of_objects, 1))], axis=1) # склеиваю по столбцам объекты-признаки с единичным признаком
print('Размерность выборки: ', X.shape)

In [ ]:
X[:5]

In [ ]:
god_like_weights = np.random.randn(number_of_features + 1) # случайные числа из random normal distribution
print('Размерность весов: ', god_like_weights.shape)

In [ ]:
X[0] @ god_like_weights

In [ ]:
y_true = X @ god_like_weights + np.random.randn(number_of_objects) * 10e-3
print('Размерность правильных ответов: ', y_true.shape)

In [ ]:
y_true[:5]

## III. Prepare for training

In [ ]:
init_weights = np.random.rand(number_of_features + 1) # .reshape(-1, 1)

In [ ]:
learning_rate = 10e-5  # гиперпараметр, скорость или шаг обучения, домножается на градиент, в диапазоне (0, 1)
stop_criterion = 10e-5 # гиперпараметр, норма разницы новых и старых весов должна быть не меньше, иначе stop GD
number_of_steps = 5000 # кол-во шагов Gradient Descent 

Предсказание для i-ого объекта:

$$
y_{\text{pred}_i} = \text{model}(x_i) = w_0 + \langle w, x_i\rangle = w_0 + w_1 * x_{i,1} + w_2 * x_{i,2} + ...
$$

Функция ошибки для i-ого объекта:

$$
\text{Loss}_i = (y_{\text{true}_i} - y_{\text{pred}_i}) ^ 2
$$

Функционал потерь - MSE:

$$
Q      = \text{mean}(\text{Loss}_i) = \text{mean}( (y_{\text{true}_i} - y_{\text{pred}_i}) ^ 2 ) = \text{mean}( (y_{\text{true}_i} - w_0 - w_1 * x_{i,1} - w_2 * x_{i,2} )^ 2)
$$

Частная производная функционала потерь по первому весу:

$$
\frac{\partial Q}{\partial w_1} = (\frac{1}{n} \cdot \sum (y_{\text{true}_i} - y_{\text{pred}_i}) ^ 2) )' = \frac{2}{n} \cdot \sum(y_{\text{true}_i} - y_{\text{pred}_i}) \cdot x_{i1}
$$

Первый вес: 
$$
\frac{2}{n} \cdot \sum (y_{\text{pred}_i} - y_{\text{true}_i}) \cdot x_{i, 1}
$$

Второй вес: 
$$
\frac{2}{n} *\cdot \sum (y_{\text{pred}_i} - y_{\text{true}_i}) \cdot x_{i, 2}
$$

Третий вес: 
$$
\frac{2}{n}*\cdot \sum (y_{\text{pred}_i} - y_{\text{true}_i}) \cdot x_{i, 3}
$$

Нулевой вес: 
$$
\frac{2}{n} *\cdot \sum (y_{\text{pred}_i} - y_{\text{true}_i}) \cdot 1
$$


## IV. Gradient Descent: First try

In [ ]:
print('Размерность нулевого объекта: ', X[0].shape)
print('Размерность весов: ', weights.shape)
print('Предсказание для нулевого объекта: ', np.sum(X[0] * weights))
print('Предсказание для нулевого объекта: ', X[0] @ weights)

In [ ]:
print('Размерность нулевого и первого объектов: ', X[0:2].shape)
print('Предсказание для нулевого и первого объекта: ', np.sum(X[0:2] * weights, axis=1))
print('Предсказание для нулевого и первого объекта: ', X[0:2] @ weights)

In [ ]:
weights = init_weights.copy()

In [ ]:
from tqdm.notebook import tqdm

loss_history = []
for step in tqdm(range(number_of_steps)):
    # 1. Предсказания
    y_pred = X @ weights

    # 2. Loss
    loss = np.mean((y_true - X @ weights) ** 2)
    loss_history.append(loss)
    
    if step % 100 == 0:
        print(f'Ошибка на шаге {step}: {loss}')

    # 3. gradient
    # loss = 1/N * sum(w_1*x_1 + w_2*x_2 + ... w_0 - y_true) ** 2
    # d_loss / d_w1 = 2/N * sum(y_pred - y_true) * x_1

    # dl_dw1 = 2/N * np.sum((y_pred - y_true) * X[:, 0])
    # dl_dw2 = 2/N * np.sum((y_pred - y_true) * X[:, 1])

    gradient = 2 / number_of_objects * (y_pred - y_true) @ X

    # 4. Update

    new_weights = weights - gradient * learning_rate

    # 5. Stop criterion

    if np.linalg.norm(new_weights - weights) < stop_criterion:
        print('Критерий остановки сработал')
        break

    weights = new_weights

In [ ]:
import matplotlib.pyplot as plt

plt.plot(loss_history)
plt.title('Loss change in Gradient Descent')
plt.xlabel('№ шага')
plt.ylabel('MSE')
plt.yscale('log')
plt.grid()
plt.show()

In [ ]:
from tqdm.notebook import tqdm


loss_history = []
# Повторить до схождения
# while True:
for step in tqdm(range(number_of_steps)):
    
    # 1. Сделать предсказание 
    # y_pred_1 = w_1 * x_11 + w_2 * x_12 + ... + w_0
    # y_pred_2 = w_1 * x_21 + w_2 * x_22 + ... + w_0
    y_prediction = X @ weights

    # 2. Посчитать ошибку MSE
    error = np.mean((y_prediction - y_true) ** 2)
    
    if step % 200 == 0: 
        print(f'Шаг {step} - MSE {round(error, 2)}')

    # 3. Посчитать производные для каждого веса
    gradient = 2 * X.T @ (y_prediction - y_true) / number_of_objects

    # 4. Сделать шаг градиентного спуска
    new_weights = weights - learning_rate * gradient 

    # 5. Проверить на критерий остановки
    if np.linalg.norm(weights - new_weights) < stop_criterion:
        print(f'Шаг {step} - Final MSE {round(error, 2)}')
        break

    weights = new_weights
    loss_history.append(error)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(np.arange(0, len(loss_history)), loss_history)
plt.title('График функции ошибки')
plt.xlabel('Шаг GD')
plt.ylabel('Значение MSE')
plt.yscale('log')
plt.grid()
plt.show()

## V. SGD

In [ ]:
weights = init_weights.copy()

In [ ]:
sgd_loss_history = []
for step in tqdm(range(number_of_steps)):
    selected_object_index = np.random.randint(0, number_of_objects)
    
    y_prediction = X @ weights # можем считать только для выбранного объекта
    error = np.mean((y_prediction - y_true) ** 2) # можем считать ошибку только для выбранного объекта
    
    if step % 200 == 0: 
        print(f'Шаг {step} - MSE {round(error, 2)}')
        
    gradient = 2 * X[selected_object_index].T * (y_prediction[selected_object_index] - y_true[selected_object_index])

    new_weights = weights - learning_rate * gradient 

    if np.linalg.norm(weights - new_weights) < stop_criterion * 10e-2:
        print(f'Шаг {step} - Final MSE {round(error, 2)}')
        break

    weights = new_weights
    sgd_loss_history.append(error)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(np.arange(0, len(loss_history)), loss_history, label='GD')
plt.plot(np.arange(0, len(sgd_loss_history)), sgd_loss_history, label='SGD')
plt.title('График функции ошибки')
plt.xlabel('Шаг GD')
plt.ylabel('Значение MSE')
plt.yscale('log')
plt.grid()
plt.legend()
plt.show()

In [ ]:
plt.plot(np.arange(0, len(loss_history)), loss_history, label='GD')
plt.plot(np.arange(0, len(sgd_loss_history)), sgd_loss_history, label='SGD')
plt.title('График функции ошибки')
plt.xlabel('Шаг GD')
plt.ylabel('Значение MSE')
plt.yscale('log')
plt.xlim((4000, 5000))
plt.grid()
plt.legend()
plt.show()